In [4]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import ElementClickInterceptedException

def get_label_value(tender, label):
    try:
        # Finds the <b> value immediately after the <span> with the label
        value = tender.find_element(
            By.XPATH, f'.//span[contains(@class,"sub_title") and contains(text(),"{label}")]/following-sibling::b[1]'
        ).text.strip()
        return value
    except:
        return ''

def safe_click(element, driver):
    try:
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
        time.sleep(0.5)
        element.click()
    except ElementClickInterceptedException:
        try:
            driver.execute_script("""
                let overlays = document.querySelectorAll('.chat-widget, .popup, .cookie-banner, .modal-backdrop');
                overlays.forEach(el => el.style.display='none');
            """)
            time.sleep(1)
        except:
            pass
        try:
            driver.execute_script("arguments[0].click();", element)
        except Exception as e:
            print("Click failed even with JS:", e)
            raise

chrome_options = Options()
chrome_options.add_argument('--start-maximized')
# chrome_options.add_argument('--headless')  # Uncomment for headless mode

driver = webdriver.Chrome(options=chrome_options)
driver.get('https://www.tendertiger.com/TenderAI/TenderAIList?searchtext=Drinking%20Water%20And%20Sanitation%20Department%20%2C%20india-tenders')
time.sleep(3)

last_count = -1
wait_attempts = 0

while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1.2)
    tenders = driver.find_elements(By.CSS_SELECTOR, 'li.tender-listing')
    current_count = len(tenders)
    print(f"Tenders loaded: {current_count}")

    try:
        view_more = driver.find_element(By.ID, "viewmorerecord")
        # Wait for the button to be enabled and visible (max 30*0.7=21s)
        for _ in range(30):
            if view_more.is_displayed() and view_more.is_enabled() and not view_more.get_attribute('disabled'):
                break
            time.sleep(0.7)
            try:
                view_more = driver.find_element(By.ID, "viewmorerecord")
            except:
                break
        if view_more.is_displayed() and view_more.is_enabled() and not view_more.get_attribute('disabled'):
            try:
                safe_click(view_more, driver)
                time.sleep(1.5)
            except Exception as e:
                print("Safe click failed, skipping:", e)
                break
        else:
            print("View More button not enabled or disappeared.")
            break
    except Exception as e:
        print("No more View More button or not clickable:", e)
        break

    if current_count == last_count:
        wait_attempts += 1
        if wait_attempts > 3:
            print("No more new tenders after multiple attempts, exiting.")
            break
    else:
        wait_attempts = 0
    last_count = current_count

# Extraction
tender_data = []
tender_elements = driver.find_elements(By.CSS_SELECTOR, 'li.tender-listing')

for tender in tender_elements:
    try:
        # Only parse if it has TID and org name
        if not tender.find_elements(By.CSS_SELECTOR, '.value.theme-color'):
            continue
        if not tender.find_elements(By.CSS_SELECTOR, 'a.org-name'):
            continue

        tid = tender.find_element(By.CSS_SELECTOR, '.value.theme-color').text.strip()
        org = tender.find_element(By.CSS_SELECTOR, 'a.org-name').text.strip()
        location = tender.find_element(By.CSS_SELECTOR, '.tender-listing-serial.value.theme-color').text.strip()
        brief = tender.find_element(By.CSS_SELECTOR, '.twm-job-address.brief a').text.strip()
        tender_value = get_label_value(tender, "Worth :")
        emd = get_label_value(tender, "EMD :")
        due_date = get_label_value(tender, "Due Date :")

        try:
            days_left = tender.find_element(By.CSS_SELECTOR, '.twm-job-post-duration').text.strip()
        except:
            days_left = ''
        try:
            detail_url = tender.find_element(By.CSS_SELECTOR, 'a.org-name').get_attribute('href')
        except:
            detail_url = ''
        tender_data.append({
            'TID': tid,
            'Organization': org,
            'Location': location,
            'Brief': brief,
            'Tender Value': tender_value,
            'EMD': emd,
            'Due Date': due_date,
            'Days Left': days_left,
            'Detail URL': detail_url
        })
    except Exception as e:
        print("Skipped a tender element due to structure:", e)

driver.quit()

df = pd.DataFrame(tender_data)
df.to_excel('tendertiger_drinking_water_sanitation_ALL_tenders.xlsx', index=False)
print(f"Scraping complete! {len(df)} valid tenders saved to Excel.")


Tenders loaded: 0
No more View More button or not clickable: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="viewmorerecord"]"}
  (Session info: chrome=138.0.7204.97); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x0x7ff7866a6f95+76917]
	GetHandleVerifier [0x0x7ff7866a6ff0+77008]
	(No symbol) [0x0x7ff786459dea]
	(No symbol) [0x0x7ff7864b0256]
	(No symbol) [0x0x7ff7864b050c]
	(No symbol) [0x0x7ff786503887]
	(No symbol) [0x0x7ff7864d84af]
	(No symbol) [0x0x7ff78650065c]
	(No symbol) [0x0x7ff7864d8243]
	(No symbol) [0x0x7ff7864a1431]
	(No symbol) [0x0x7ff7864a21c3]
	GetHandleVerifier [0x0x7ff78697d2cd+3051437]
	GetHandleVerifier [0x0x7ff786977923+3028483]
	GetHandleVerifier [0x0x7ff7869958bd+3151261]
	GetHandleVerifier [0x0x7ff7866c185e+185662]
	GetHandleVerifier [0x0x7ff7866c971f+218111]
	GetHandleVerifier [0x0x7

In [5]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def get_label_value(tender, label):
    try:
        value = tender.find_element(
            By.XPATH, f'.//span[contains(@class,"sub_title") and contains(text(),"{label}")]/following-sibling::b[1]'
        ).text.strip()
        return value
    except:
        return ''

def safe_click(element, driver):
    try:
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
        time.sleep(0.5)
        element.click()
    except ElementClickInterceptedException:
        try:
            driver.execute_script("""
                let overlays = document.querySelectorAll('.chat-widget, .popup, .cookie-banner, .modal-backdrop');
                overlays.forEach(el => el.style.display='none');
            """)
            time.sleep(1)
        except:
            pass
        try:
            driver.execute_script("arguments[0].click();", element)
        except Exception as e:
            print("Click failed even with JS:", e)
            raise

chrome_options = Options()
chrome_options.add_argument('--start-maximized')
# chrome_options.add_argument('--headless')  # Uncomment for headless mode

# Optional: Set user agent (may help with anti-bot)
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36')

driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 25)

driver.get('https://www.tendertiger.com/TenderAI/TenderAIList?searchtext=Drinking%20Water%20And%20Sanitation%20Department%20%2C%20india-tenders')

time.sleep(5)  # Give time for popups to show up if any
driver.save_screenshot('after_load.png')
print("Screenshot saved as after_load.png.")

# Try to close any basic modal/popup
try:
    close_buttons = driver.find_elements(By.CSS_SELECTOR, '.close, .modal .btn-close, [aria-label="Close"], .modal-footer .btn')
    for btn in close_buttons:
        try:
            btn.click()
            print("Closed a popup/modal.")
            time.sleep(1)
        except Exception as e:
            print("Failed to click close button:", e)
except Exception as e:
    print("No popup found or unable to close:", e)

# Wait for at least one tender to appear (or timeout after 25s)
try:
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'li.tender-listing')))
except Exception as e:
    driver.save_screenshot('no_listing_debug.png')
    print("No listing found; see no_listing_debug.png for what loaded.")
    driver.quit()
    exit("No tenders loaded; check screenshot and page source to debug.")

last_count = -1
wait_attempts = 0

while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1.2)
    tenders = driver.find_elements(By.CSS_SELECTOR, 'li.tender-listing')
    current_count = len(tenders)
    print(f"Tenders loaded: {current_count}")

    try:
        view_more = driver.find_element(By.ID, "viewmorerecord")
        for _ in range(30):
            if view_more.is_displayed() and view_more.is_enabled() and not view_more.get_attribute('disabled'):
                break
            time.sleep(0.7)
            try:
                view_more = driver.find_element(By.ID, "viewmorerecord")
            except:
                break
        if view_more.is_displayed() and view_more.is_enabled() and not view_more.get_attribute('disabled'):
            try:
                safe_click(view_more, driver)
                time.sleep(1.5)
            except Exception as e:
                print("Safe click failed, skipping:", e)
                break
        else:
            print("View More button not enabled or disappeared.")
            break
    except Exception as e:
        print("No more View More button or not clickable:", e)
        break

    if current_count == last_count:
        wait_attempts += 1
        if wait_attempts > 3:
            print("No more new tenders after multiple attempts, exiting.")
            break
    else:
        wait_attempts = 0
    last_count = current_count

# Extraction
tender_data = []
tender_elements = driver.find_elements(By.CSS_SELECTOR, 'li.tender-listing')

for tender in tender_elements:
    try:
        if not tender.find_elements(By.CSS_SELECTOR, '.value.theme-color'):
            continue
        if not tender.find_elements(By.CSS_SELECTOR, 'a.org-name'):
            continue

        tid = tender.find_element(By.CSS_SELECTOR, '.value.theme-color').text.strip()
        org = tender.find_element(By.CSS_SELECTOR, 'a.org-name').text.strip()
        location = tender.find_element(By.CSS_SELECTOR, '.tender-listing-serial.value.theme-color').text.strip()
        brief = tender.find_element(By.CSS_SELECTOR, '.twm-job-address.brief a').text.strip()
        tender_value = get_label_value(tender, "Worth :")
        emd = get_label_value(tender, "EMD :")
        due_date = get_label_value(tender, "Due Date :")

        try:
            days_left = tender.find_element(By.CSS_SELECTOR, '.twm-job-post-duration').text.strip()
        except:
            days_left = ''
        try:
            detail_url = tender.find_element(By.CSS_SELECTOR, 'a.org-name').get_attribute('href')
        except:
            detail_url = ''
        tender_data.append({
            'TID': tid,
            'Organization': org,
            'Location': location,
            'Brief': brief,
            'Tender Value': tender_value,
            'EMD': emd,
            'Due Date': due_date,
            'Days Left': days_left,
            'Detail URL': detail_url
        })
    except Exception as e:
        print("Skipped a tender element due to structure:", e)

driver.quit()

df = pd.DataFrame(tender_data)
df.to_excel('tendertiger_drinking_water_sanitation_ALL_tenders.xlsx', index=False)
print(f"Scraping complete! {len(df)} valid tenders saved to Excel.")


Screenshot saved as after_load.png.
No listing found; see no_listing_debug.png for what loaded.


MaxRetryError: HTTPConnectionPool(host='localhost', port=49164): Max retries exceeded with url: /session/412ffe6ee71d515279bc7f2bae2774cf/execute/sync (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000018C63E395B0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [6]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def get_label_value(tender, label):
    try:
        value = tender.find_element(
            By.XPATH, f'.//span[contains(@class,"sub_title") and contains(text(),"{label}")]/following-sibling::b[1]'
        ).text.strip()
        return value
    except:
        return ''

def safe_click(element, driver):
    try:
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
        time.sleep(0.5)
        element.click()
    except ElementClickInterceptedException:
        try:
            driver.execute_script("""
                let overlays = document.querySelectorAll('.chat-widget, .popup, .cookie-banner, .modal-backdrop');
                overlays.forEach(el => el.style.display='none');
            """)
            time.sleep(1)
        except:
            pass
        try:
            driver.execute_script("arguments[0].click();", element)
        except Exception as e:
            print("Click failed even with JS:", e)
            raise

chrome_options = Options()
chrome_options.add_argument('--start-maximized')
# chrome_options.add_argument('--headless')  # Uncomment to run headless mode

# Optional: randomize or spoof user-agent to reduce block risk
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 25)

driver.get('https://www.tendertiger.com/TenderAI/TenderAIList?searchtext=Drinking%20Water%20And%20Sanitation%20Department%20%2C%20india-tenders')

# Optional: handle popup
time.sleep(5)
try:
    close_buttons = driver.find_elements(By.CSS_SELECTOR, '.close, .modal .btn-close, [aria-label="Close"], .modal-footer .btn')
    for btn in close_buttons:
        try:
            btn.click()
            print("Closed a popup/modal.")
            time.sleep(1)
        except Exception as e:
            print("Failed to click close button:", e)
except Exception as e:
    print("No popup found or unable to close:", e)

# Wait for at least one tender to appear
try:
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'li.tender-listing')))
except Exception as e:
    driver.save_screenshot('no_listing_debug.png')
    print("No listing found; see no_listing_debug.png for what loaded.")
    print(driver.page_source[:5000])
    driver.quit()
    exit("No tenders loaded; check screenshot and page source to debug.")

last_count = -1
wait_attempts = 0

while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1.2)
    tenders = driver.find_elements(By.CSS_SELECTOR, 'li.tender-listing')
    current_count = len(tenders)
    print(f"Tenders loaded: {current_count}")

    try:
        view_more = driver.find_element(By.ID, "viewmorerecord")
        for _ in range(30):
            if view_more.is_displayed() and view_more.is_enabled() and not view_more.get_attribute('disabled'):
                break
            time.sleep(0.7)
            try:
                view_more = driver.find_element(By.ID, "viewmorerecord")
            except:
                break
        if view_more.is_displayed() and view_more.is_enabled() and not view_more.get_attribute('disabled'):
            try:
                safe_click(view_more, driver)
                time.sleep(1.5)
            except Exception as e:
                print("Safe click failed, skipping:", e)
                break
        else:
            print("View More button not enabled or disappeared.")
            break
    except Exception as e:
        print("No more View More button or not clickable:", e)
        break

    if current_count == last_count:
        wait_attempts += 1
        if wait_attempts > 3:
            print("No more new tenders after multiple attempts, exiting.")
            break
    else:
        wait_attempts = 0
    last_count = current_count

# Extraction
tender_data = []
tender_elements = driver.find_elements(By.CSS_SELECTOR, 'li.tender-listing')

for tender in tender_elements:
    try:
        if not tender.find_elements(By.CSS_SELECTOR, '.value.theme-color'):
            continue
        if not tender.find_elements(By.CSS_SELECTOR, 'a.org-name'):
            continue

        tid = tender.find_element(By.CSS_SELECTOR, '.value.theme-color').text.strip()
        org = tender.find_element(By.CSS_SELECTOR, 'a.org-name').text.strip()
        location = tender.find_element(By.CSS_SELECTOR, '.tender-listing-serial.value.theme-color').text.strip()
        brief = tender.find_element(By.CSS_SELECTOR, '.twm-job-address.brief a').text.strip()
        tender_value = get_label_value(tender, "Worth :")
        emd = get_label_value(tender, "EMD :")
        due_date = get_label_value(tender, "Due Date :")

        try:
            days_left = tender.find_element(By.CSS_SELECTOR, '.twm-job-post-duration').text.strip()
        except:
            days_left = ''
        try:
            detail_url = tender.find_element(By.CSS_SELECTOR, 'a.org-name').get_attribute('href')
        except:
            detail_url = ''
        tender_data.append({
            'TID': tid,
            'Organization': org,
            'Location': location,
            'Brief': brief,
            'Tender Value': tender_value,
            'EMD': emd,
            'Due Date': due_date,
            'Days Left': days_left,
            'Detail URL': detail_url
        })
    except Exception as e:
        print("Skipped a tender element due to structure:", e)

driver.quit()

df = pd.DataFrame(tender_data)
df.to_excel('tendertiger_drinking_water_sanitation_ALL_tenders.xlsx', index=False)
print(f"Scraping complete! {len(df)} valid tenders saved to Excel.")


No popup found or unable to close: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=138.0.7204.97)
Stacktrace:
	GetHandleVerifier [0x0x7ff7866a6f95+76917]
	GetHandleVerifier [0x0x7ff7866a6ff0+77008]
	(No symbol) [0x0x7ff786459dea]
	(No symbol) [0x0x7ff786445f15]
	(No symbol) [0x0x7ff78646abf4]
	(No symbol) [0x0x7ff7864dfa85]
	(No symbol) [0x0x7ff7864fff72]
	(No symbol) [0x0x7ff7864d8243]
	(No symbol) [0x0x7ff7864a1431]
	(No symbol) [0x0x7ff7864a21c3]
	GetHandleVerifier [0x0x7ff78697d2cd+3051437]
	GetHandleVerifier [0x0x7ff786977923+3028483]
	GetHandleVerifier [0x0x7ff7869958bd+3151261]
	GetHandleVerifier [0x0x7ff7866c185e+185662]
	GetHandleVerifier [0x0x7ff7866c971f+218111]
	GetHandleVerifier [0x0x7ff7866afb14+112628]
	GetHandleVerifier [0x0x7ff7866afcc9+113065]
	GetHandleVerifier [0x0x7ff786696c98+10616]
	BaseThreadInitThunk [0x0x7ffa0b75dbe7+23]
	RtlUserThreadStart [0x0x7ffa0d845

InvalidSessionIdException: Message: invalid session id
Stacktrace:
	GetHandleVerifier [0x0x7ff7866a6f95+76917]
	GetHandleVerifier [0x0x7ff7866a6ff0+77008]
	(No symbol) [0x0x7ff786459c1c]
	(No symbol) [0x0x7ff7864a055f]
	(No symbol) [0x0x7ff7864d8332]
	(No symbol) [0x0x7ff7864d2e53]
	(No symbol) [0x0x7ff7864d1f19]
	(No symbol) [0x0x7ff786424b05]
	GetHandleVerifier [0x0x7ff78697d2cd+3051437]
	GetHandleVerifier [0x0x7ff786977923+3028483]
	GetHandleVerifier [0x0x7ff7869958bd+3151261]
	GetHandleVerifier [0x0x7ff7866c185e+185662]
	GetHandleVerifier [0x0x7ff7866c971f+218111]
	(No symbol) [0x0x7ff786423b00]
	GetHandleVerifier [0x0x7ff786a95f38+4201496]
	BaseThreadInitThunk [0x0x7ffa0b75dbe7+23]
	RtlUserThreadStart [0x0x7ffa0d845a4c+44]


In [7]:
import pandas as pd
from bs4 import BeautifulSoup

with open('your_downloaded_file.html', encoding='utf-8') as f:
    soup = BeautifulSoup(f, 'html.parser')

tenders = soup.select('li.tender-listing')
data = []

def get_label_value_bs4(tender, label):
    tag = tender.find('span', class_='sub_title', string=lambda s: s and label in s)
    if tag and tag.find_next('b'):
        return tag.find_next('b').get_text(strip=True)
    return ''

for tender in tenders:
    try:
        tid = tender.find('b', class_='value theme-color').get_text(strip=True)
        org = tender.find('a', class_='org-name').get_text(strip=True)
        location = tender.find('b', class_='tender-listing-serial value theme-color').get_text(strip=True)
        brief = tender.select_one('.twm-job-address.brief a').get_text(strip=True)
        tender_value = get_label_value_bs4(tender, 'Worth :')
        emd = get_label_value_bs4(tender, 'EMD :')
        due_date = get_label_value_bs4(tender, 'Due Date :')
        try:
            days_left = tender.find('span', class_='twm-job-post-duration').get_text(strip=True)
        except:
            days_left = ''
        try:
            detail_url = tender.find('a', class_='org-name')['href']
        except:
            detail_url = ''
        data.append({
            'TID': tid,
            'Organization': org,
            'Location': location,
            'Brief': brief,
            'Tender Value': tender_value,
            'EMD': emd,
            'Due Date': due_date,
            'Days Left': days_left,
            'Detail URL': detail_url
        })
    except Exception as e:
        print("Error parsing tender:", e)

df = pd.DataFrame(data)
df.to_excel('tenders_from_saved_html.xlsx', index=False)
print(f"Parsed {len(df)} tenders from saved HTML!")


FileNotFoundError: [Errno 2] No such file or directory: 'your_downloaded_file.html'